**German Credit Dataset**

# 02 - Fairness Groups Analysis (Revisado)

**Objectives**
- Identify and define sensitive attributes related to fairness
- Specify the favorable and unfavorable label values in the target variable
- Define privileged and unprivileged groups to be used in fairness analysis

In [60]:
import pandas as pd
import numpy as np

## 1. Load Data

In [61]:
file_path = '../data/processed/german_df_processed_1.csv'
df = pd.read_csv(file_path, sep=r',', header=0)

In [62]:
df

,checking_account_status,duration_months,credit_history,purpose,credit_amount,savings_account_status,employment_status,installment_rate,marriage_status_sex,guarantors,...,property,age,other_debts,housing,existing_credits_count,job,dependents,own_telephone?,foreign_worker?,good_client?
0,1,6,5,4,1169,5,5,4,3,1,...,1,67,3,2,2,3,1,1,1,1
1,2,48,3,4,5951,1,3,2,2,1,...,1,22,3,2,1,3,1,0,1,0
2,4,12,5,7,2096,1,4,2,3,1,...,1,49,3,2,1,2,2,0,1,1
3,1,42,3,3,7882,1,4,2,3,3,...,2,45,3,3,1,3,2,0,1,1
4,1,24,4,1,4870,1,3,3,3,1,...,4,53,3,3,2,3,2,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,4,12,3,3,1736,1,4,3,2,1,...,1,31,3,2,1,2,1,0,1,1
996,1,30,3,2,3857,1,3,4,1,1,...,2,40,3,2,1,4,1,1,1,1
997,4,12,3,4,804,1,5,4,3,1,...,3,38,3,2,1,3,1,0,1,1
998,1,45,3,4,1845,1,3,4,3,1,...,4,23,3,3,1,3,1,1,1,0


## 2. Identifying Sensitive Attr. and Privileged/Unprivileged Groups

In [63]:
df_eng = df.copy()

### 2.1. marriage_status_sex

The column "marriage_status_sex" encodes both gender and marital status simultaneously, with the following mapping:
    1: "male divorced/separated"
    2: "female divorced/separated/married"
    3: "male single"
    4: "male married/widowed"
    5: "female single"
For simplicity, a new column "sex" will be created to indicate only the individual's gender.

The "marriage_status_sex" column will then be dropped.

In [64]:
sex_map = {
    1: 1,  # male
    2: 2,  # female
    3: 1,  # male
    4: 1,  # male
    5: 2,  # female
}

df_eng['sex'] = df_eng['marriage_status_sex'].map(sex_map)
df_eng = df_eng.drop('marriage_status_sex', axis=1)

In [65]:
counts_sex = df_eng['sex'].value_counts()
percent_sex = (counts_sex / len(df_eng)) * 100

print("Sex Distribution")
print(f"Male (1): {counts_sex.get(1)} instances ({percent_sex.get(1):.1f}% of total)")
print(f"Female (2): {counts_sex.get(2)} instances ({percent_sex.get(2):.1f}% of total)")

Sex Distribution
Male (1): 690 instances (69.0% of total)
Female (2): 310 instances (31.0% of total)


Since 69% of the individuals in the dataset are male, males will be considered the privileged group and females the unprivileged group.

### 2.2. foreign_worker?

In [66]:
counts_foreign = df_eng['foreign_worker?'].value_counts()
percent_foreign = (counts_foreign / len(df_eng)) * 100

print("Foreign Worker Distribution")
print(f"Foreign (True): {counts_foreign.get(1)} instances ({percent_foreign.get(1):.1f}% of total)")
print(f"Local (False): {counts_foreign.get(0)} instances ({percent_foreign.get(0):.1f}% of total)")

Foreign Worker Distribution
Foreign (True): 963 instances (96.3% of total)
Local (False): 37 instances (3.7% of total)


Since 96% of the individuals in the dataset are foreign, foreigns will be considered the privileged group and locals the unprivileged group.

### 2.3. age



In [67]:
AGE_THRESHOLD = 25
df_eng['age_cat'] = np.where(df_eng['age'] < AGE_THRESHOLD, 2, 1)
counts_age = df_eng['age_cat'].value_counts()
percent_age = (counts_age / len(df_eng)) * 100

print("Age Distribution")
print(f"age < 25: {counts_age.get(2)} instances ({percent_age.get(2):.1f}% of total)")
print(f"age >= 25: {counts_age.get(1)} instances ({percent_age.get(1):.1f}% of total)")

df_eng = df_eng.drop('age', axis=1)

Age Distribution
age < 25: 149 instances (14.9% of total)
age >= 25: 851 instances (85.1% of total)


Individuals aged less than 25 years are considered the unprivileged group, consistent with the standard used in the academic fairness literature for the German Credit Dataset.

## 3. Fairness Variable Definitions

In [68]:
TARGET_COLUMN = 'good_client?'
FAVORABLE_LABEL = 1   # 'Good client'
UNFAVORABLE_LABEL = 0 # 'No good client'

SENSITIVE_ATTR = 'sex'
PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Male
UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # Female

# SENSITIVE_ATTR = 'foreign_worker?'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # Foreign
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 0}] # Local

# SENSITIVE_ATTR = 'age_cat'
# PRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 1}]   # age >= 25
# UNPRIVILEGED_GROUPS = [{SENSITIVE_ATTR: 2}] # age < 25 

## 4. Exporting Dataset

In [69]:
file_out_path = '../data/processed'

df_eng.to_csv(file_out_path + '/german_df_processed_2.csv', index=False)